# B5: grouped-query attention (GQA)

**Goal:** replace the HF head-expansion helper used in B4 with our own code.

TinyLlama has 32 query heads, 4 KV heads and head dimension 64. Think of 32 people asking different questions while sharing 4 reference books. Each group of 8 queries shares a book, not its answers.

**Spec:** Q `(Nq, Hq, D)`, K/V `(Nk, Hkv, D)`, boolean allowed mask `(Nq, Nk)` -> output `(Nq, Hq, D)` and weights `(Hq, Nq, Nk)`. Hq must be divisible by Hkv. KV heads are repeated **consecutively**, not cyclically.

Run top to bottom in **Colab on a T4**. No runtime results are assumed before this run.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


In [ ]:
import importlib
import torch
import tinyserve.models.gqa, tinyserve.probe
for module in (tinyserve.models.gqa, tinyserve.probe):
    importlib.reload(module)
from tinyserve.models.gqa import expand_kv_heads, grouped_query_attention
from tinyserve.models.attention import causal_mask
from tinyserve.envinfo import collect_env, save_result

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())


## 1. Test first, including B4 regression checks

Test exact head mapping and HF expansion parity, then compare GQA with separate FP64 runs per query head. Cover MHA (no sharing), GQA (groups) and MQA (one KV head for everyone).

B4 tests run too because the real-model probe now uses our GQA. A failure stops the notebook. Read any skips; HF and CUDA tests should be available here.

In [ ]:
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b4_attention.py", "tests/test_b5_gqa.py", "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. Which KV head serves which query?

Use head IDs as the tensor values so the mapping is visible:

```
queries  0..7  -> KV 0
queries  8..15 -> KV 1
queries 16..23 -> KV 2
queries 24..31 -> KV 3
```

`repeat_interleave` repeats each head consecutively. A cyclic `[0,1,2,3,0,1,2,3,...]` pattern would connect the wrong learned heads.

In [ ]:
kv_head_ids = torch.arange(4, device=DEVICE, dtype=torch.float32).reshape(1, 4, 1)
expanded_ids = expand_kv_heads(kv_head_ids, 32)
head_mapping = expanded_ids[0, :, 0].int().tolist()
print("KV head used by each query head:", head_mapping)
assert head_mapping == [0] * 8 + [1] * 8 + [2] * 8 + [3] * 8
print("original shape:", tuple(kv_head_ids.shape), "expanded shape:", tuple(expanded_ids.shape))


## 3. Shared K/V, different attention weights

Two queries, `+1` and `-1`, share keys `[0, 2]` and values `[10, 20]`. The positive query favors the second key; the negative query favors the first. Sharing K/V does **not** force equal scores or outputs.

Both keys are allowed in this tiny rectangular example. It isolates head sharing, not causality.

In [ ]:
hand_query = torch.tensor([[[1.0], [-1.0]]], device=DEVICE)
hand_key = torch.tensor([[[0.0]], [[2.0]]], device=DEVICE)
hand_value = torch.tensor([[[10.0]], [[20.0]]], device=DEVICE)
hand_output, hand_weights = grouped_query_attention(hand_query, hand_key, hand_value,
                                                    torch.ones(1, 2, device=DEVICE, dtype=torch.bool))
print("query +1 weights:", hand_weights[0, 0].tolist())
print("query -1 weights:", hand_weights[1, 0].tolist())
print("outputs:", hand_output[0, :, 0].tolist())
assert hand_output[0, 0, 0] > hand_output[0, 1, 0]


## 4. TinyLlama shapes and FP16 comparison

Q `(5, 32, 64)`, K/V `(5, 4, 64)` -> output `(5, 32, 64)` and weights `(32, 5, 5)`.

Compare FP16 to FP32 using the same quantized inputs. B4 computes in FP32 and rounds the final output once; `rtol=atol=5e-4` covers that rounding. Future weights must still be zero.

In [ ]:
demo_generator = torch.Generator(device=DEVICE).manual_seed(17)
demo_query = torch.randn(5, 32, 64, generator=demo_generator, device=DEVICE).half()
demo_key = torch.randn(5, 4, 64, generator=demo_generator, device=DEVICE).half()
demo_value = torch.randn(5, 4, 64, generator=demo_generator, device=DEVICE).half()
demo_mask = causal_mask(torch.arange(5, device=DEVICE))
demo_output, demo_weights = grouped_query_attention(demo_query, demo_key, demo_value, demo_mask)
float_output, _ = grouped_query_attention(demo_query.float(), demo_key.float(), demo_value.float(), demo_mask)
fp16_max_diff = (demo_output.float() - float_output).abs().max().item()
future_weight_max = demo_weights.masked_select(~demo_mask[None]).max().item()
print("Q:", tuple(demo_query.shape), "K/V:", tuple(demo_key.shape))
print("output:", tuple(demo_output.shape), "weights:", tuple(demo_weights.shape))
print("FP16 vs FP32 max difference:", fp16_max_diff, "future weight:", future_weight_max)
torch.testing.assert_close(demo_output.float(), float_output, rtol=5e-4, atol=5e-4)
assert future_weight_max == 0.0


## 5. Why the cache is 8 times smaller

Cache storage follows **KV heads**, not query heads:

`2 (K/V) * 22 layers * KV heads * 64 dimensions * 2 bytes`.

GQA: 22 KiB/token and 44 MiB per 2048-token request. MHA: 176 KiB/token and 352 MiB per request.

**Important:** this reference temporarily expands K/V to run B4's math. The future cache stores only the original 4 heads. A fast backend later reads shared heads directly. GQA does not remove 7 out of 8 queries: all 32 heads still compute their own scores.

In [ ]:
gqa_bytes_per_token = 2 * 22 * 4 * 64 * 2
mha_bytes_per_token = 2 * 22 * 32 * 64 * 2
print(f"GQA: {gqa_bytes_per_token / 1024:.0f} KiB/token; MHA: {mha_bytes_per_token / 1024:.0f} KiB/token")
print(f"2048 tokens: GQA {gqa_bytes_per_token * 2048 / 2**20:.0f} MiB; MHA {mha_bytes_per_token * 2048 / 2**20:.0f} MiB")
print("cache reduction factor:", mha_bytes_per_token // gqa_bytes_per_token)
assert mha_bytes_per_token == 8 * gqa_bytes_per_token


## 6. Our RoPE + GQA + attention against real HF activations

Load TinyLlama in FP32 and use eager attention. Compare outputs before `o_proj` in layers 0, 10 and 21. The probe now uses **our own head expansion**, not HF's helper.

The probe asserts `rtol=atol=1e-5`. This verifies the attention core, not the whole decoder. Close other GPU-heavy notebook kernels if memory is low (FP32 weights take about 4.4 GB).

In [ ]:
from tinyserve.basics import load_model, encode
from tinyserve.probe import compare_attention_layers
tokenizer, model = load_model(device=DEVICE, dtype=torch.float32)
model.set_attn_implementation("eager")
prompt_ids = encode(tokenizer, "The capital of France is", DEVICE)
attention_rows = compare_attention_layers(model, prompt_ids)
for row in attention_rows:
    print(f"layer {row['layer']:>2}: max |diff| {row['max_abs_diff']:.2e}, future weight {row['future_weight_max']:.1f}")


## 7. Save results, including the test summary

The min/max range is across tested layers, not performance trials. Copy the printed JSON back to the repo after the run so validation remains recorded.

In [ ]:
layer_differences = [row["max_abs_diff"] for row in attention_rows]
saved_path = save_result("B5_gqa", {
    "query_heads": 32, "kv_heads": 4, "head_dim": 64,
    "head_mapping": head_mapping,
    "shared_kv_hand_outputs": hand_output[0, :, 0].tolist(),
    "fp16_vs_fp32_max_abs_diff": fp16_max_diff,
    "future_weight_max": future_weight_max,
    "gqa_cache_bytes_per_token": gqa_bytes_per_token,
    "mha_cache_bytes_per_token": mha_bytes_per_token,
    "real_model_dtype": "float32", "real_model_backend": "eager",
    "real_model_comparison": attention_rows,
    "real_model_diff_range_across_layers": {"min": min(layer_differences), "max": max(layer_differences)},
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


## 8. Three interview questions

1. **MHA vs GQA vs MQA?** One KV head per query, one per group, or one for all queries.
2. **Why does cache size shrink but output size stay the same?** Storage follows KV head count; output follows query head count.
3. **Is this a zero-copy paged kernel?** No. This readable reference uses temporary expanded K/V. A later optimized kernel reads compact KV directly.

**Next: B6 MLP (SwiGLU).** Two expanding projections, `silu(gate) * up`, then a down projection. It accounts for about 69% of TinyLlama's weights.

After validation: retain results, update the tracker and guide, then commit and push.

In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
